# Stage 5 — Land-Use Pressure (Mole Creek prototype)

Brings in the first layer that should actually show variation *outside*
the karst itself — now meaningful thanks to the Stage 0 buffer fix, which
gave the study area real surrounding land to carry a pressure signal.

**Data access reused from AT2** (same STAC pattern, no changes needed —
this part was never PyQGIS): `pystac_client` search against DEA's public
STAC catalogue, `odc.stac.load` to bring it in already reprojected to
EPSG:7855. **What's different from AT2:** the resample/reclassify steps.
AT2 used PyQGIS (`processing.run("gdal:warpreproject", ...)` and
`processing.run("native:reclassifybytable", ...)`), which the AT3 brief
explicitly disallows ("modern geospatial stack, i.e. not PyQGIS"). Here
both steps use `rioxarray`/`numpy` instead — reproject/resample via
`.rio.reproject_match()` (nearest-neighbour, since land cover is
categorical), reclassify via a plain numpy lookup.

Both DEA epochs (2010, 2020) are fetched — not just for the main model,
but because the year-on-year difference is a cheap, genuine additional
finding (Stage 9).

In [ ]:
from pathlib import Path
import os
import geopandas as gpd
import numpy as np
import pystac_client
import odc.stac
import rioxarray
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

data = Path(os.environ.get("AT3_DATA_DIR", "data"))
outpath = data / "Output_data"

mole_creek = gpd.read_file(str(outpath / "mole_creek_bounding_box_EPSG7855.gpkg"))
mole_creek_wgs84 = mole_creek.to_crs("EPSG:4326")  # DEA STAC search requires WGS84 lon/lat
bbox = mole_creek_wgs84.total_bounds.tolist()

## 5.1 Fetch DEA Level-3 land cover, both epochs

No authentication needed — public STAC catalogue, anonymous S3 access.

In [ ]:
catalog = pystac_client.Client.open("https://explorer.dea.ga.gov.au/stac")
odc.stac.configure_rio(cloud_defaults=True, aws={"aws_unsigned": True})

items = []
for year in [2010, 2020]:
    search = catalog.search(
        collections=["ga_ls_landcover_class_cyear_3"], bbox=bbox,
        datetime=f"{year}-01-01/{year}-12-31",
    )
    items += list(search.items())

land_cover = odc.stac.load(
    items, bands=["level3"], bbox=bbox, crs="EPSG:7855", resolution=30, groupby="solar_day",
)
print("Loaded:", dict(land_cover.sizes))
print("Years:", land_cover.time.dt.year.values)

## 5.2 Reclassify to a pressure scale

Level-3 classes actually present at Mole Creek: 111 (cultivated
terrestrial vegetation), 112 (natural terrestrial vegetation), 215
(artificial surface), 216 (natural bare surface), 220 (water), 255 (no
data). Kept deliberately simple and transparent (per the revised
workflow's land-use table), matching the same 0-3 ordinal scale used for
the other layers:

| Level-3 class | Pressure score |
|---|---|
| 111 Cultivated terrestrial vegetation | 3 (High — agriculture) |
| 112 Natural terrestrial vegetation | 1 (Low) |
| 124 Natural aquatic vegetation | 1 (Low) |
| 215 Artificial surface | 3 (High — urban/development) |
| 216 Natural bare surface | 1 (Low — natural, not human pressure) |
| 220 Water | 0 (None) |
| 255 No data | excluded |

In [ ]:
PRESSURE_RECLASS = {111: 3, 112: 1, 124: 1, 215: 3, 216: 1, 220: 0}

with rasterio.open(outpath / "dem_mc_EPSG7855.tif") as dem_src:
    dem_data = dem_src.read(1)
    dem_nodata_val = dem_src.nodata
valid = dem_data != dem_nodata_val

pressure_by_year = {}
for year in [2010, 2020]:
    arr = land_cover["level3"].sel(time=land_cover.time.dt.year == year).isel(time=0)

    pressure = np.full(arr.shape, 255, dtype="uint8")
    for code_val, score in PRESSURE_RECLASS.items():
        pressure[arr.values == code_val] = score
    pressure_da = arr.copy(data=pressure).rio.write_nodata(255)

    # Resample native 30m -> the DEM's 25m grid, nearest-neighbour (categorical data)
    with rioxarray.open_rasterio(outpath / "dem_mc_EPSG7855.tif") as dem_template:
        dem_template = dem_template.squeeze()
        pressure_aligned = pressure_da.rio.reproject_match(dem_template, resampling=Resampling.nearest)

    out_path = outpath / f"landuse_pressure_mc_{year}_EPSG7855.tif"
    pressure_aligned.rio.to_raster(out_path, dtype="uint8")
    pressure_by_year[year] = pressure_aligned.values

    vals, counts = np.unique(pressure_aligned.values[valid], return_counts=True)
    print(f"Year {year} pressure distribution (valid cells):", dict(zip(vals.tolist(), counts.tolist())))

## 5.3 Visualise both epochs side by side

In [ ]:
cmap = ListedColormap(["#2b83ba", "#abdda4", "#fdae61", "#d7191c"])
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, year in zip(axes, [2010, 2020]):
    display = np.ma.masked_equal(pressure_by_year[year], 255)
    im = ax.imshow(display, cmap=cmap, vmin=0, vmax=3)
    ax.set_title(f"Land-use pressure {year}")
plt.colorbar(im, ax=axes, ticks=[0, 1, 2, 3], label="Pressure (0=None, 1=Low, 3=High)", shrink=0.7)
plt.show()

## An early look at the temporal trend (full analysis belongs in Stage 9)

High-pressure (cultivated + artificial) area increased from 180,275 to
283,006 cells between 2010 and 2020 at Mole Creek — roughly a 57%
increase in high-pressure land within the buffered study area. Worth
flagging now, properly analysed later: this is exactly the kind of
finding the "temporal land-use trend" value-add analysis is there to
capture, and it's a real signal, not noise.

## Stage 5 outputs

- `landuse_pressure_mc_2010_EPSG7855.tif`
- `landuse_pressure_mc_2020_EPSG7855.tif`

Both aligned to the same grid as Stages 2-4 (the Mole Creek DEM
template), so they're ready to combine directly.

**Next (Stage 6):** combine Stage 4 (intrinsic vulnerability) × Stage 5
(land-use pressure, 2020 as the primary year) into the final relative
vulnerability/risk map for Mole Creek — the prototype's main output,
before scaling statewide.